In [ ]:
!pip install -q datasets transformers accelerate bitsandbytes faiss-cpu xformers peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:

# ✅ IMPORTS

from datasets import load_dataset
from PIL import Image
from io import BytesIO
import requests
import torch
import numpy as np
import faiss
import random
import torch.nn.functional as F
from collections import Counter
from transformers import (
    BitsAndBytesConfig, BlipProcessor, BlipForConditionalGeneration,
    SiglipProcessor, SiglipModel,
    AutoProcessor, LlavaForConditionalGeneration
)


# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16
# )




random.seed(42)

# ✅ JS Divergence
def js_divergence(p, q):
    p = torch.tensor(p, dtype=torch.float32)
    q = torch.tensor(q, dtype=torch.float32)
    p = torch.nn.functional.softmax(p, dim=0)
    q = torch.nn.functional.softmax(q, dim=0)
    m = 0.5 * (p + q)
    kl_pm = torch.nn.functional.kl_div(m.log(), p, reduction='sum')
    kl_qm = torch.nn.functional.kl_div(m.log(), q, reduction='sum')
    return 0.5 * (kl_pm + kl_qm).item()

# ✅ BLIP captioner
class LightweightBLIPCaptioner:
    def __init__(self, model_name="Salesforce/blip-image-captioning-base", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = BlipProcessor.from_pretrained(model_name)
        self.model = BlipForConditionalGeneration.from_pretrained(model_name).to(self.device)

    def caption(self, image):
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=64)
        return self.processor.batch_decode(out, skip_special_tokens=True)[0]

# ✅ SigLIP retriever
class SigLIPRetriever:
    def __init__(self, model_name="google/siglip-base-patch16-224", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = SiglipProcessor.from_pretrained(model_name)
        self.model = SiglipModel.from_pretrained(model_name).to(self.device)
        self.text_data, self.image_data, self.image_captions = [], [], []
        self.embeddings, self.metadata = [], []
        self.index = None
        self.captioner = None

    def load_auto_captioner(self, model_name="Salesforce/blip-image-captioning-base"):
        self.captioner = LightweightBLIPCaptioner(model_name)

    def encode_texts(self, texts):
        inputs = self.processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(self.device)
        with torch.no_grad():
            feats = self.model.get_text_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats.cpu().numpy()

    def encode_images(self, images):
        inputs = self.processor(images=images, return_tensors="pt").to(self.device)
        with torch.no_grad():
            feats = self.model.get_image_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats.cpu().numpy()

    def add_texts(self, texts):
        for text in texts:
            self.text_data.append(text)
            emb = self.encode_texts([text])[0]
            self.embeddings.append(emb)
            self.metadata.append(("text", len(self.text_data) - 1))

    def add_images(self, images):
        for img in images:
            if img is None:
                continue
            self.image_data.append(img)
            img_idx = len(self.image_data) - 1
            img_emb = self.encode_images([img])[0]
            self.embeddings.append(img_emb)
            self.metadata.append(("image", img_idx))

    def build_index(self):
        if self.embeddings:
            all_embs = np.vstack(self.embeddings).astype("float32")
            self.index = faiss.IndexFlatIP(all_embs.shape[1])
            self.index.add(all_embs)

    def encode_query(self, query_text=None, query_image=None):
        if query_text and query_image:
            text_emb = self.encode_texts([query_text])
            image_emb = self.encode_images([query_image])
            combined = text_emb + image_emb
            return combined / np.linalg.norm(combined, axis=1, keepdims=True)
        elif query_text:
            return self.encode_texts([query_text])
        elif query_image:
            return self.encode_images([query_image])
        else:
            raise ValueError("Provide text and/or image as query.")

    def retrieve(self, query_text=None, query_image=None, top_k=10, max_kl_threshold=0.4):
        query_emb = self.encode_query(query_text, query_image)
        scores, indices = self.index.search(query_emb, top_k)
        context_texts, context_images = [], []
        for i in indices[0]:
            doc_type, doc_idx = self.metadata[i]
            if doc_type == "text":
                doc_emb = self.encode_texts([self.text_data[doc_idx]])[0]
            elif doc_type == "image":
                doc_emb = self.encode_images([self.image_data[doc_idx]])[0]
            else:
                continue

            divergence = js_divergence(query_emb[0], doc_emb)
            if divergence <= max_kl_threshold:
                if doc_type == "text":
                    context_texts.append(self.text_data[doc_idx])
                elif doc_type == "image":
                    context_texts.append("[Image]")
                    context_images.append(self.image_data[doc_idx])
        return context_texts, context_images

# ✅ Multimodal Generator
# quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
class MultimodalGenerator:
    def __init__(self, model_name="llava-hf/llava-1.5-7b-hf"):
        self.model_name = model_name
        self.processor = AutoProcessor.from_pretrained(self.model_name)
        self.model = LlavaForConditionalGeneration.from_pretrained(
            self.model_name,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            # quantization_config=quantization_config,
            device_map="auto"
        )

    def generate(self, query, query_image=None, context_texts=[], context_images=None):
        img_list=[]

        if query_image:
          question_string=f"<image>\n{query}"
          # img_list.append(query_image)
        else:
          question_string=query
        # selected_image = context_images[0] if context_images else None
        context = "\n".join(context_texts)
        if context_images:
          img_place_holder="<image>"*len(context_images)
          context_string=f"{img_place_holder}\n{context}"
          [img_list.append(im_ctx) for im_ctx in context_images]
        else:
          context_string=context


        if query_image:
          img_list.append(query_image)
        prompt_text = f"Context:{context_string}\n\nQuestion: {question_string}\n"
        prompt=f"USER: {prompt_text}\nASSISTANT:"
        if img_list:
          inputs = self.processor(img_list, prompt, padding=True, return_tensors="pt").to("cuda")
        else:
          inputs = self.processor(prompt, padding=True, return_tensors="pt").to("cuda")
        output = self.model.generate(**inputs, max_new_tokens=200)
        generated_text = self.processor.batch_decode(output, skip_special_tokens=True)


        return generated_text[0].split("ASSISTANT:")[-1]


# ✅ RAG Wrapper
class MultimodalRAG:
    def __init__(self, k=None):
        self.retriever = SigLIPRetriever()
        self.retriever.load_auto_captioner()
        self.generator = MultimodalGenerator()

    def add_texts(self, texts):
        self.retriever.add_texts(texts)

    def add_images(self, images):
        self.retriever.add_images(images)

    def build_index(self):
        self.retriever.build_index()

    def query(self, query_text=None, query_image=None, top_k=10, max_kl_threshold=0.5):
        ctx_texts, ctx_images = self.retriever.retrieve(query_text=query_text, query_image=query_image, top_k=top_k, max_kl_threshold=max_kl_threshold)
        return self.generator.generate(query=query_text, query_image=query_image, context_texts=ctx_texts, context_images=ctx_images)




In [ ]:
k_ar=[3,5,10,15,20]
rag_acc_l=[]
non_rag_acc_l=[]

In [ ]:
train_data = load_dataset("facebook/textvqa", split="validation[:150]")
train_questions = train_data["question"]
train_answers = train_data["answers"]
train_image_urls = train_data["flickr_300k_url"]
train_question_ids = train_data["question_id"]




train_images, rag_texts = [], []
for q, a, url in zip(train_questions, train_answers, train_image_urls):
    try:
        img = Image.open(BytesIO(requests.get(url).content)).convert("RGB")
        train_images.append(img)
        rag_texts.append(f"Q: {q}\nA: {a[0] if a else ''}")
    except Exception as e:
        print("⚠️ Skipping:", url)

# ✅ Initialize and build RAG
rag = MultimodalRAG()
rag.add_texts(rag_texts)
rag.add_images(train_images)
rag.build_index()
gen = MultimodalGenerator()



# ✅ Select 100 random questions from train[:500] for evaluation
indices = random.sample(range(len(train_questions)), 100)
test_questions = [train_questions[i] for i in indices]
test_answers = [train_answers[i] for i in indices]
test_images = []
for i in indices:
    try:
        img = Image.open(BytesIO(requests.get(train_image_urls[i]).content)).convert("RGB")
    except:
        img = None
    test_images.append(img)


for k in k_ar:


  # ✅ Evaluate with RAG
  rag_preds, rag_refs = [], []
  for q, img, gt, qid in zip(test_questions, test_images, test_answers, [train_question_ids[i] for i in indices]):
      result = rag.query(query_text=q, query_image=img, top_k=k)
      rag_preds.append({"question_id": qid, "answer": result.strip()})
      rag_refs.append({"question_id": qid, "answers": {"answer": gt}})


  # ✅ Evaluate without context (no RAG)
  noctx_preds = []
  for q, img, qid in zip(test_questions, test_images, [train_question_ids[i] for i in indices]):
      result = gen.generate(query=q, query_image=img, context_texts=[], context_images=[])
      noctx_preds.append({"question_id": qid, "answer": result.strip()})

  # ✅ Shared references for both
  references = [{"question_id": qid, "answers": {"answer": ans}} for qid, ans in zip([train_question_ids[i] for i in indices], test_answers)]

  # ✅ VQA-style soft accuracy
  def compute_vqa_accuracy(predictions, references):
      total, correct = 0, 0
      for pred, ref in zip(predictions, references):
          if not pred or "answer" not in pred or not ref or "answers" not in ref or "answer" not in ref["answers"]:
              continue
          pred_ans = pred["answer"].strip().lower()
          gt_answers = [ans.strip().lower() for ans in ref["answers"]["answer"] if ans]
          if not gt_answers:
              continue
          match_count = Counter(gt_answers)[pred_ans]
          accuracy = min(1.0, match_count / 3)
          correct += accuracy
          total += 1
      return correct / total if total > 0 else 0.0

  rag_acc = compute_vqa_accuracy(rag_preds, references)
  noctx_acc = compute_vqa_accuracy(noctx_preds, references)
  print(f"for k is {k}")
  rag_acc_l.append(rag_acc*100)
  non_rag_acc_l.append(noctx_acc*100)
  print(f"✅ RAG Accuracy: {rag_acc*100:.2f}%")
  print(f"✅ No-RAG Accuracy: {noctx_acc*100:.2f}%")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

textvqa.py:   0%|          | 0.00/5.02k [00:00<?, ?B/s]

The repository for facebook/textvqa contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/facebook/textvqa.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] Y


Generating train split:   0%|          | 0/34602 [00:00<?, ? examples/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

You may have used the wrong order for inputs. `images` should be passed before `text`. The `images` and `text` inputs will be swapped. This behavior will be deprecated in transformers v4.47.


for k is 3
✅ RAG Accuracy: 9.67%
✅ No-RAG Accuracy: 1.00%
for k is 5
✅ RAG Accuracy: 10.67%
✅ No-RAG Accuracy: 1.00%
for k is 10
✅ RAG Accuracy: 13.00%
✅ No-RAG Accuracy: 1.00%
for k is 15
✅ RAG Accuracy: 14.00%
✅ No-RAG Accuracy: 1.00%
for k is 20
✅ RAG Accuracy: 17.67%
✅ No-RAG Accuracy: 1.00%


In [ ]:
k_ar

[3, 5, 10, 15, 20]

In [ ]:
non_rag_acc_l

[1.0, 1.0, 1.0, 1.0, 1.0]

In [ ]:
rag_acc_l

[9.666666666666666,
 10.666666666666666,
 13.0,
 14.000000000000002,
 17.666666666666668]